**INSTALLAZIONE DELLE LIBRERIE**

In [ ]:
!pip uninstall -y bitsandbytes
!pip install -q -U transformers accelerate peft qwen-vl-utils torchvision bitsandbytes

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2


**TESTO IL MODELLO**

In [ ]:
mock_input_from_retriever = {
    "sub_claim": "La vitamina D riduce il rischio di infezione da COVID-19",
    "evidence_passages": [
        {
            "id": "PUBMED_123",
            "text": "Uno studio randomizzato controllato su 5000 pazienti non ha mostrato una riduzione significativa dell'incidenza di COVID-19 con l'integrazione di Vitamina D.",
            "source": "Journal of Internal Medicine, 2022"
        },
        {
            "id": "PMC_456",
            "text": "Sebbene la carenza di Vitamina D sia associata a casi più gravi, non ci sono prove che l'integrazione prevenga l'infezione iniziale.",
            "source": "Nature Reviews, 2021"
        }
    ]
}

**inizializzazione agente**

In [ ]:
import os
import torch

os.environ["BNB_CUDA_VERSION"] = "121"
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

class ReasoningAgent:
    def __init__(self, model_id="Qwen/Qwen2-VL-7B-Instruct"):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
            attn_implementation="sdpa"
        )

    def generate_cot(self, sub_claim, evidence_list, image_path=None):
        evidence_text = "\n".join([f"- {e['text']} (Fonte: {e['source']})" for e in evidence_list])

        prompt_text = f"""Sei un esperto di fact-checking biomedico. Analizza il seguente claim basandoti sulle evidenze fornite e sull'immagine allegata.
Genera un ragionamento passo-passo (Chain-of-Thought) che colleghi le prove al claim, evidenziando eventuali concordanze o contraddizioni.

CLAIM: {sub_claim}

EVIDENZE SCIENTIFICHE:
{evidence_text}

RAGIONAMENTO LOGICO:"""

        user_content = []
        if image_path:
            user_content.append({"type": "image", "image": image_path})
        user_content.append({"type": "text", "text": prompt_text})

        messages = [
            {"role": "system", "content": [{"type": "text", "text": "Sei un analista scientifico rigoroso e imparziale."}]},
            {"role": "user", "content": user_content}
        ]

        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(self.model.device)

        generated_ids = self.model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True
        )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        response = self.processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        return response.strip()

agent = ReasoningAgent()
reasoning = agent.generate_cot(
    mock_input_from_retriever["sub_claim"],
    mock_input_from_retriever["evidence_passages"],
    image_path= None #QUI DOVREBBE ANDARE IL PERCORSO DELL'IMMAGINE
)
print(reasoning)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Passo 1: Identificare il claim in questione.
Il claim a cui si riferisce è: "La vitamina D riduce il rischio di infezione da COVID-19".

Passo 2: Recuperare le evidenze fornite.
Le evidenze fornite includono:
1. Un studio randomizzato controllato su 5000 pazienti che non ha mostrato una riduzione significativa dell'incidenza di COVID-19 con l'integrazione di Vitamina D.
2. La carenza di Vitamina D è associata a casi più gravi, ma non ci sono prove che l'integrazione prevenga l'infezione iniziale.

Passo 3: Analizzare le evidenze in relazione al claim.
Il primo studio randomizzato controllato suggerisce che l'integrazione di Vitamina D non riduce significativamente il rischio di infezione da COVID-19. Questo è in contrasto con il claim che afferma che la vitamina D riduce il rischio di infezione da COVID-19.

Passo 4: Confrontare le evidenze con il claim.
Il secondo punto, che indica l'associazione tra la carenza di Vitamina D e casi più gravi, non sostiene direttamente il claim che la 

**VERACITY**

In [ ]:
import os
os.environ["BNB_CUDA_VERSION"] = "121"
os.environ['LD_LIBRARY_PATH'] += f":{os.path.dirname(__import__('torch').__file__)}/lib"

import torch
import gc
import json
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, pipeline
from qwen_vl_utils import process_vision_info

class ReasoningAndVeracityPipeline:
    def __init__(self, reasoning_model_id="Qwen/Qwen2-VL-7B-Instruct", veracity_model_id="cross-encoder/nli-deberta-v3-base"):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.reasoning_processor = AutoProcessor.from_pretrained(reasoning_model_id)
        self.reasoning_model = Qwen2VLForConditionalGeneration.from_pretrained(
            reasoning_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
            attn_implementation="sdpa"
        )

        self.veracity_classifier = pipeline(
            "text-classification",
            model=veracity_model_id,
            device=self.reasoning_model.device,
            torch_dtype=torch.float16,
            truncation=True,
            max_length=512
        )





    def _generate_reasoning(self, claim, evidence_list, image_path=None):
        evidence_text = "\n".join([f"- {e['testo']}" for e in evidence_list])

        system_prompt = """Sei un analista biomedico neutrale. Il tuo compito è spiegare la relazione logica tra un claim e le evidenze.
REGOLE CRITICHE:
1. MASSIMA SINTESI: Il ragionamento DEVE essere di 2-3 frasi (sotto i 150 token).
2. NEUTRALITÀ ASSOLUTA: Spiega solo come le informazioni si allineano o si scontrano. NON usare MAI parole conclusive come "smentisce", "conferma", "è falso" o "è vero".
3. NIENTE PREAMBOLI: Vai dritto all'analisi dei fatti."""

        user_prompt_text = f"CLAIM: {claim}\nEVIDENZE SCIENTIFICHE:\n{evidence_text}\nRAGIONAMENTO LOGICO:"

        user_content = []
        if image_path:
            user_content.append({"type": "image", "image": image_path})
        user_content.append({"type": "text", "text": user_prompt_text})

        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "text", "text": "CLAIM: L'aglio cura il cancro.\nEVIDENZE SCIENTIFICHE:\n- Nessuno studio clinico dimostra l'efficacia dell'aglio contro le neoplasie.\nRAGIONAMENTO LOGICO:"}]},
            {"role": "assistant", "content": [{"type": "text", "text": "Il claim postula un'azione curativa dell'aglio contro i tumori. Tuttavia, le evidenze riportano un'assenza totale di studi clinici che provino tale efficacia, indicando una netta discrepanza tra l'affermazione e i dati scientifici disponibili."}]},
            {"role": "user", "content": [{"type": "text", "text": "CLAIM: L'aspirina abbassa la temperatura corporea.\nEVIDENZE SCIENTIFICHE:\n- L'acido acetilsalicilico agisce come antipiretico inibendo le prostaglandine.\nRAGIONAMENTO LOGICO:"}]},
            {"role": "assistant", "content": [{"type": "text", "text": "Il claim descrive l'abbassamento della temperatura. Le evidenze definiscono il principio attivo come 'antipiretico', un meccanismo farmacologico che ha l'effetto diretto di contrastare la febbre e ridurre il calore corporeo. I due concetti sono semanticamente sovrapponibili."}]},
            {"role": "user", "content": user_content}
        ]

        text = self.reasoning_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        model_inputs = self.reasoning_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(self.reasoning_model.device)

        with torch.no_grad():
            generated_ids = self.reasoning_model.generate(
                **model_inputs,
                max_new_tokens=200,
                temperature=0.1,
                do_sample=True
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.reasoning_processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

        del model_inputs
        del generated_ids
        del generated_ids_trimmed
        torch.cuda.empty_cache()

        return response.strip()


    def _assess_veracity(self, claim, evidence_list, reasoning_text):
        evidence_all = " ".join([e["testo"] for e in evidence_list])
        context = f"Scientific Evidence: {evidence_all} Analysis: {reasoning_text}"
        hypothesis = f"Based on this, it is true that {claim}"

        result = self.veracity_classifier([{"text": context, "text_pair": hypothesis}])[0]

        raw_label = result["label"].lower()
        score = result["score"]

        del context
        del hypothesis
        del result
        torch.cuda.empty_cache()
        gc.collect()

        if raw_label == "entailment" or raw_label == "label_0":
            final_label = "Supported"
        elif raw_label == "contradiction" or raw_label == "label_2":
            final_label = "Refuted"
        else:
            final_label = "Not Enough Information"

        return final_label, round(score, 4)

    def process_claim(self, input_data):
        claim = input_data.get("sub_claim", "")
        evidence = input_data.get("evidence_passages", [])
        image_path = input_data.get("image_path", None)

        chain_of_thought = self._generate_reasoning(claim, evidence, image_path)
        verdict, confidence = self._assess_veracity(claim, evidence, chain_of_thought)

        final_output = {
            "claim": claim,
            "verdict": verdict,
            "confidence_score": confidence,
            "chain_of_thought_log": chain_of_thought,
            "supporting_evidence": evidence
        }

        return final_output

    def print_readable_report(self, verdetto_json):
        print("\n" + "="*60)
        print("  REPORT MEDFACTCHECK: ANALISI COMPLETATA")
        print("="*60)
        print(f" AFFERMAZIONE: {verdetto_json['claim']}")
        percentuale_sicurezza = verdetto_json['confidence_score'] * 100
        print(f" VERDETTO FINALE: {verdetto_json['verdict']} (Sicurezza: {percentuale_sicurezza:.1f}%)")
        print("\n RAGIONAMENTO LOGICO:")
        print(verdetto_json['chain_of_thought_log'])
        print("\n FONTI SCIENTIFICHE CONSULTATE:")

        lista_fonti = verdetto_json['supporting_evidence']
        for numero, fonte in enumerate(lista_fonti, 1):
            titolo = fonte.get('titolo', 'N/D')
            testo = fonte.get('testo', '')
            url = fonte.get('url', 'N/D')
            print(f"  {numero}. Titolo: {titolo}")
            print(f"     Testo: {testo}")
            print(f"     Link: {url}\n")
        print("="*60)

mock_input = {
    "sub_claim": "I vaccini a mRNA alterano il DNA umano.",
    "image_path": None,
    "evidence_passages": [
        {
            "titolo": "Studio sui meccanismi dell'mRNA",
            "url": "https://pubmed.ncbi.nlm.nih.gov/esempio1/",
            "testo": "L'mRNA dei vaccini non entra nel nucleo della cellula dove si trova il DNA, rendendo impossibile un'alterazione del genoma umano."
        },
        {
            "titolo": "Sicurezza dei vaccini Covid-19",
            "url": "https://pubmed.ncbi.nlm.nih.gov/esempio2/",
            "testo": "Non ci sono prove cliniche o biologiche che l'RNA messaggero possa integrarsi nel DNA dell'ospite."
        }
    ]
}

my_pipeline = ReasoningAndVeracityPipeline()
risultato = my_pipeline.process_claim(mock_input)

print("\n--- JSON GREZZO PER IL DATABASE ---")
print(json.dumps(risultato, indent=2, ensure_ascii=False))
my_pipeline.print_readable_report(risultato)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]


--- JSON GREZZO PER IL DATABASE ---
{
  "claim": "I vaccini a mRNA alterano il DNA umano.",
  "verdict": "Refuted",
  "confidence_score": 0.8303,
  "chain_of_thought_log": "Il claim afferma che i vaccini a mRNA possono alterare il DNA umano. Le evidenze, invece, sottolineano che l'mRNA non raggiunge il nucleo cellulare, dove si trova il DNA, e non ci sono prove che l'RNA messaggero possa integrarsi nel DNA dell'ospite. Queste informazioni sembrano contraddire il claim.",
  "supporting_evidence": [
    {
      "titolo": "Studio sui meccanismi dell'mRNA",
      "url": "https://pubmed.ncbi.nlm.nih.gov/esempio1/",
      "testo": "L'mRNA dei vaccini non entra nel nucleo della cellula dove si trova il DNA, rendendo impossibile un'alterazione del genoma umano."
    },
    {
      "titolo": "Sicurezza dei vaccini Covid-19",
      "url": "https://pubmed.ncbi.nlm.nih.gov/esempio2/",
      "testo": "Non ci sono prove cliniche o biologiche che l'RNA messaggero possa integrarsi nel DNA dell'ospite